# Transaction Encoding
## Project: Product Placement Optimisation
### Purpose: Transform clean transaction data into basket format for Apriori and FP-Growth algorithms
## Student: Samikshya Baniya
## Student ID: 230360
## Module: ST6001CEM Individual Project
### Input: data/processed/sales_data_cleaned.csv
### Output: Encoded transaction matrix ready for market basket analysis

## Why Transaction Encoding?

The clean data has one row per product per invoice.
Association rule algorithms need one row per basket with 
True/False values for each product.

This step transforms:
- FROM: long format (one row per product)
- TO: wide format (one row per basket, one column per category)

## Step 1: Load Clean Data

In [1]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv(
    r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\sales_data_cleaned.csv'
)

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Invoices: {df['invoice_no'].nunique():,}")
print(f"Categories: {df['category'].nunique()}")

Data loaded successfully!
Shape: (767180, 14)
Invoices: 218,037
Categories: 25


## Step 2: Create Category Baskets

In [2]:
basket_categories = df.groupby('invoice_no')['category'].apply(list)

print(f"Total baskets: {len(basket_categories):,}")
print(f"\nSample basket 1:")
print(basket_categories.iloc[0])
print(f"\nSample basket 2:")
print(basket_categories.iloc[1])

Total baskets: 218,037

Sample basket 1:
['FOOD STAPLES', 'FROZEN FOODS']

Sample basket 2:
['COOKING OIL']


### What these baskets tell us

Sample basket 1 has two categories, Sample basket 2 has only one. Most baskets in this store are small, which we already saw in notebook 03. A single category basket means the customer came in for one specific thing and left. These customers are harder to influence with placement. The multi-category baskets are where placement strategy can increase basket value.

## Step 3: Apply Transaction Encoder

In [3]:
te = TransactionEncoder()
te_array = te.fit(basket_categories).transform(basket_categories)

basket_df = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction encoding complete!")
print(f"Shape: {basket_df.shape}")
print(f"Rows = baskets: {basket_df.shape[0]:,}")
print(f"Columns = categories: {basket_df.shape[1]}")
print(f"\nFirst 3 rows:")
basket_df.head(3)

Transaction encoding complete!
Shape: (218037, 25)
Rows = baskets: 218,037
Columns = categories: 25

First 3 rows:


,ALCOHOLIC BEVERAGES,BABY CARE,BAKERY,BISCUITS AND COOKIES,BREAKFAST CEREALS,CANNED AND PACKAGED FOODS,CIGARETTE AND TOBACCO,CLEANING SUPPLIES,CONFECTIONERY,COOKING OIL,...,HOUSEHOLD ITEMS,NAMKEEN AND SNACKS,NOODLES,PARTY SUPPLIES,PERSONAL CARE,POOJA ITEMS,RICE,SOFT DRINKS AND JUICES,STATIONERY,TEA AND SPICES
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Step 4: Save Encoded Transaction Matrix

In [4]:
import os

output_path = r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\basket_encoded.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)
basket_df.to_csv(output_path, index=False)

print("Encoded basket data saved!")
print(f"Location: {output_path}")
print(f"Shape: {basket_df.shape}")
print(f"\nCategory columns:")
for col in basket_df.columns:
    true_count = basket_df[col].sum()
    pct = (true_count / len(basket_df)) * 100
    print(f"  {col:35s} {true_count:,} baskets ({pct:.1f}%)")

Encoded basket data saved!
Location: D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\basket_encoded.csv
Shape: (218037, 25)

Category columns:
  ALCOHOLIC BEVERAGES                 8,622 baskets (4.0%)
  BABY CARE                           3,899 baskets (1.8%)
  BAKERY                              5,911 baskets (2.7%)
  BISCUITS AND COOKIES                33,209 baskets (15.2%)
  BREAKFAST CEREALS                   5,349 baskets (2.5%)
  CANNED AND PACKAGED FOODS           62,371 baskets (28.6%)
  CIGARETTE AND TOBACCO               11,199 baskets (5.1%)
  CLEANING SUPPLIES                   30,944 baskets (14.2%)
  CONFECTIONERY                       30,750 baskets (14.1%)
  COOKING OIL                         32,926 baskets (15.1%)
  DAIRY PRODUCTS                      26,083 baskets (12.0%)
  ELECTRICAL SUPPLIES                 309 baskets (0.1%)
  FOOD STAPLES                        91,988 baskets (42.2%)
  FROZEN FOODS                        9,

## Transaction Encoding Complete!

### Summary:
- Input: 767,180 transaction rows
- Output: 218,037 x 25 encoded basket matrix
- Each row = one complete shopping trip
- Each column = one product category
- True = category was purchased, False = was not

### Key finding:
FOOD STAPLES appears in 42.2% of all baskets making it
the anchor category for placement strategy.

### Next Step: 05_market_basket_analysis.ipynb